### Section 1 - Series & DataFrame fundamentals

In [79]:
import tempfile
import pathlib
import numpy as np
import pandas as pd

assert pd.__version__.startswith("3."), pd.__version__

def make_books() -> pd.DataFrame:
    return pd.DataFrame({
        "title": ["Dune", "Neuromancer", "Foundation", "Hyperion",
                  "The Left Hand of Darkness", "Snow Crash"],
        "author": ["Herbert", "Gibson", "Asimov", "Simmons", "Le Guin", "Stephenson"],
        "year": [1965, 1984, 1951, 1989, 1969, 1992],
        "pages": [412, 271, 255, 482, 304, 480],
        "price": [18.99, 14.50, 12.00, 17.25, 15.75, 16.40],
    })

In [80]:
# Exercise 1.1
# A
s1 = pd.Series([1, 2, 3], index=["a", "b", "c"])
s2 = pd.Series([10, 20, 30], index=["b", "c", "d"])

r = s1 + s2                  # predict: ["a", "b", "c", "d"], [NaN, 12, 23, NaN], float64
r_np = s1 + s2.to_numpy()    # predict: ["a", "b", "c"], [11, 22, 33], int64
print(r)
print(r_np)
# r is float because of NaN, and r_np is int since numpy array has no labels so there is nothing to align. Values are paired by position
# in r values are paired by labels, while in r_np they are paired by position

# B
filled = s1.add(s2, fill_value=0)
assert filled.to_dict() == {"a": 1.0, "b": 12.0, "c": 23.0, "d": 30.0}
assert filled.dtype == "float64"

a     NaN
b    12.0
c    23.0
d     NaN
dtype: float64
a    11
b    22
c    33
dtype: int64


In [81]:
# Exercise 1.2
books = make_books()

# A) Add "price_per_100_pages": price per 100 pages, rounded to 2 decimals.
books["price_per_100_pages"] = (books["price"] / books["pages"] * 100).round(2)
assert books["price_per_100_pages"].tolist() == [4.61, 5.35, 4.71, 3.58, 5.18, 3.42]
# B) Make a new frame with "title" as the index.
by_title = books.set_index("title")

assert by_title.loc["Dune", "year"] == 1965
assert by_title.shape == (6, 5)

# C)
type(books["price"]) # Series
type(books[["price"]]) # DataFrame
books.shape # (6, 6)

type(books[["title", "price"]]) # DataFrame
books["price"] # Error

0    18.99
1    14.50
2    12.00
3    17.25
4    15.75
5    16.40
Name: price, dtype: float64

In [82]:
# Exercise 1.3
# A
make_books().dtypes # str, str, int64, int64, float64 - text columns get str, McKinney's book show object
# B
books = make_books()
numeric = books.select_dtypes(include="number")
text = books.select_dtypes(include="str")

assert list(numeric.columns) == ["year", "pages", "price"]
assert list(text.columns) == ["title", "author"]
assert books["title"].dtype == "str"

make_books().to_numpy().dtype # object
make_books()[["year", "price"]].to_numpy().dtype # float64

dtype('float64')

In [83]:
# Exercise 1.4
books = make_books()
newest_first = books.sort_values("year", ascending=False)
top2 = books.nlargest(2, "price")
assert newest_first["title"].tolist() == ["Snow Crash", "Hyperion", "Neuromancer",
                                          "The Left Hand of Darkness", "Dune", "Foundation"]
assert top2["title"].tolist() == ["Dune", "Hyperion"]

books["pages_rank"] = books["pages"].rank(ascending=False)
books
assert books.set_index("title")["pages_rank"].to_dict() == {
    "Dune": 3.0, "Neuromancer": 5.0, "Foundation": 6.0, "Hyperion": 1.0,
    "The Left Hand of Darkness": 4.0, "Snow Crash": 2.0}

desc = books["price"].describe() # describe() returns: count, mean, std, min, 25%, 50%, 75%, max
assert desc["max"] == 18.99 and desc["count"] == 6
desc

count     6.000000
mean     15.815000
std       2.398739
min      12.000000
25%      14.812500
50%      16.075000
75%      17.037500
max      18.990000
Name: price, dtype: float64

In [84]:
# Exercise 1.5
books = make_books()
books["published"] = pd.to_datetime(books["year"].astype(str) + "-01-01")
books["format"] = pd.Categorical(["hardcover", "paperback", "paperback",
                                  "hardcover", "paperback", "ebook"])

with tempfile.TemporaryDirectory() as tmp:
    csv_path = pathlib.Path(tmp) / "books.csv"
    parquet_path = pathlib.Path(tmp) / "books.parquet"

    books.to_csv(csv_path, index=False)
    books.to_parquet(parquet_path)

    from_csv = pd.read_csv(csv_path)
    from_parquet = pd.read_parquet(parquet_path)

    print(from_csv.dtypes, from_parquet.dtypes, sep="\n\n") # str, str / datetime,  category

    assert pd.api.types.is_string_dtype(from_csv["published"])
    assert pd.api.types.is_string_dtype(from_csv["format"])
    assert pd.api.types.is_datetime64_any_dtype(from_parquet["published"])
    assert isinstance(from_parquet["format"].dtype, pd.CategoricalDtype)

    # B) Read the CSV again so that "published" comes back as a datetime.
    fixed = pd.read_csv(csv_path, parse_dates=["published"])

    assert pd.api.types.is_datetime64_any_dtype(fixed["published"])
    # Parquet, because it stores exact data types

title            str
author           str
year           int64
pages          int64
price        float64
published        str
format           str
dtype: object

title                   str
author                  str
year                  int64
pages                 int64
price               float64
published    datetime64[us]
format             category
dtype: object


In [85]:
s = pd.Series([300, 450, 300, 200])
s.rank(ascending=False)
# It shows float64 as dtype since it is trying to predict data type to avoid dtype changing dynamically

0    2.5
1    1.0
2    2.5
3    4.0
dtype: float64